# Solar Filament V11.3 — reproducible pipeline walkthrough

**Final highest scoring version: V4 3-fold YOLOv8m-seg ensemble + image-guided Trim1.** Displayed Kaggle Public LB **0.39**. This notebook documents the entire workflow. Training and inference require local MAGFiLO data / three weights and are **disabled by default**. The OOF validation shown is one model per image, **not ensemble validation**.

In [ ]:
from pathlib import Path
import subprocess, json, hashlib
PROJECT_ROOT = Path.cwd().resolve()
TRAIN_DATA_AVAILABLE=False
RUN_PREPARE=False
RUN_TRAIN=False
RUN_OOF=False
RUN_TEST_INFERENCE=False
TEST_DIR=PROJECT_ROOT / "MAGFiLO_1.0_Kaggle_2026/test/test_images"
FOLD_WEIGHTS=[PROJECT_ROOT / f"runs/segment/runs_v4/yolov8m_patch_v4_fold{i}/weights/best.pt" for i in range(3)]

def run_cmd(args):
    print("RUN:"," ".join(map(str,args)))
    subprocess.run([str(x) for x in args],check=True)
print("Project:",PROJECT_ROOT)

## 1. Data and fold assignment
Original 2048 H-alpha images; 707 physical training images / 1,154 COCO annotation records. Split at physical-image level (multi-annotator records stay together). Train/val folds are generated with `make_3fold_split.py`.

In [ ]:
if RUN_PREPARE and TRAIN_DATA_AVAILABLE:
    run_cmd(["python", "code/solar_filament_v4/scripts/make_3fold_split.py", "--manifest", "dataset_yolo_v2_native/manifest.csv", "--output", "dataset_yolo_v4_3fold_manifest.csv"])
    for i in range(3):
        run_cmd(["python", "code/solar_filament_v4/scripts/prepare_fold_patches.py", "--manifest", "dataset_yolo_v4_3fold_manifest.csv", "--fold",i,"--output-root","dataset_v4_folds","--tile-size",1024,"--stride",768])

## 2. Three fold YOLOv8m-seg training
Seeds 42,43,44; 60 epochs max; early stopping patience 20; batch 4; `imgsz=1024`. All models initialized from `yolov8m-seg.pt`.

In [ ]:
if RUN_TRAIN and TRAIN_DATA_AVAILABLE:
    for i in range(3):
        run_cmd(["python","code/solar_filament_v4/scripts/train_3fold.py","--fold",i,"--data-root","dataset_v4_folds","--epochs",60,"--patience",20,"--batch",4,"--imgsz",1024,"--device",0,"--project","runs_v4"])

## 3. Development validation: HONEST single-model OOF
Do NOT run three-model ensemble on train records and present it as OOF. Fold-specific validation uses only the held-out fold model. Validation measures selected checkpoint development performance, not a never-touched final test.

In [ ]:
if RUN_OOF and TRAIN_DATA_AVAILABLE:
    assert all(p.exists() for p in FOLD_WEIGHTS)
    run_cmd(["python","code/solar_filament_v4/scripts/evaluate_oof_pq.py","--models",*FOLD_WEIGHTS,"--manifest","dataset_yolo_v4_3fold_manifest.csv","--tile-size",1024,"--stride",768,"--imgsz",1024,"--conf",0.30,"--merge-iou",0.10,"--merge-containment",0.45,"--max-instances",15,"--duplicate-iou",0.80,"--duplicate-containment",0.88,"--output","outputs/eval_v4_oof.csv"])

## 4. Test ensemble → Trim1 → compressed RLE submission
First run the V4 three-model ensemble on 180 full-resolution test images. Apply the **same frozen Trim1 rule** only after exclusive pixel ownership. Each row in final CSV is one filament.

In [ ]:
if RUN_TEST_INFERENCE:
    assert TEST_DIR.is_dir() and all(p.exists() for p in FOLD_WEIGHTS)
    run_cmd(["bash","scripts/run_final_pipeline.sh",TEST_DIR,*FOLD_WEIGHTS])
else:
    print("Skipped GPU inference. Enable RUN_TEST_INFERENCE after supplying data and best.pt.")

## 5. Quantitative evidence and checksum checks
Fold summaries below are the supplied **single-model OOF** results. Kaggle Public LB 0.39 is documented separately by screenshot and final audit hash.

In [ ]:
s=json.loads((PROJECT_ROOT/"results/v11_2_pooled_3fold_oof.json").read_text())
print("OOF baseline PQ:",round(s["baseline"]["pq"],6))
print("OOF + Trim1 PQ:",round(s["trim1"]["pq"],6))
a=json.loads((PROJECT_ROOT/"results/v11_3_trim1_audit.json").read_text())
print("V11.3 recorded SHA256:",a["output_sha256"])
print("Changed masks:",a["masks_changed"],"no overlaps:",a["number_of_output_overlaps"])

## 6. Submission preparation and official validation
The audited final 0.39 CSV is included at the repository root as `submission_v11_3_trim1.csv`. Confirm its SHA256 before use. The official self-evaluation notebook is available at https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook .

In [ ]:
expected="13c19935e917018fe2a5c550927c2dc1fdf317987be4dba563e22e756d217f8f"
final_file=PROJECT_ROOT/"submission_v11_3_trim1.csv"
if final_file.exists():
    actual=hashlib.sha256(final_file.read_bytes()).hexdigest()
    print("EXACT 0.39 CSV:",actual==expected,actual)
else:
    print("Missing final CSV. Obtain from original server before report handoff.")